In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

print(" Podyum Operasyonu: Logit-Space Optimized Stacking Başlatıldı...")

# ==========================================
# 1. DOSYALARI OTOMATİK TESPİT EDİP YÜKLEME
# ==========================================
train_path, test_path, sub_path = None, None, None
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'train.csv': train_path = os.path.join(root, file)
        elif file == 'test.csv': test_path = os.path.join(root, file)
        elif file == 'sample_submission.csv': sub_path = os.path.join(root, file)

if not train_path:
    train_path = 'train.csv' if os.path.exists('train.csv') else None
    test_path = 'test.csv' if os.path.exists('test.csv') else None
    sub_path = 'sample_submission.csv' if os.path.exists('sample_submission.csv') else None

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sub_path)

# ==========================================
# 2. METİNSEL SÜTUNUN ENKODE EDİLMESI
# ==========================================
gender_map = {'male': 1, 'female': 0}
train['feat_50'] = train['feat_50'].map(gender_map)
test['feat_50'] = test['feat_50'].map(gender_map)

X = train.drop(columns=['ID', 'class'])
y = train['class']
X_test = test.drop(columns=['ID'])

# ==========================================
# 3. VERİ ÖLÇEKLENDİRME
# ==========================================
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

pos_weight = (len(y) - sum(y)) / sum(y)

# ==========================================
# 4. MULTI-SEED 10-FOLD OOF DÖNGÜSÜ
# ==========================================
seeds = [42, 2026]
n_splits = 10

meta_preds_ensemble = np.zeros(len(X_test))
pure_lgb_preds = np.zeros(len(X_test))
pure_cat_preds = np.zeros(len(X_test))

for seed in seeds:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    oof_train = np.zeros((len(X_scaled), 3))
    oof_test = np.zeros((len(X_test_scaled), 3))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
        X_train, y_train = X_scaled.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_scaled.iloc[val_idx], y.iloc[val_idx]
        
        # 1. LightGBM
        lgb = LGBMClassifier(
            n_estimators=2500, learning_rate=0.012, num_leaves=63,
            subsample=0.8, colsample_bytree=0.8, random_state=seed,
            is_unbalance=True, n_jobs=-1, verbose=-1
        )
        lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[])
        lgb_fold_pred = lgb.predict_proba(X_test_scaled)[:, 1]
        oof_train[val_idx, 0] = lgb.predict_proba(X_val)[:, 1]
        oof_test[:, 0] += lgb_fold_pred / n_splits
        pure_lgb_preds += lgb_fold_pred / (n_splits * len(seeds))
        
        # 2. XGBoost
        xgb = XGBClassifier(
            n_estimators=2500, learning_rate=0.012, max_depth=8,
            subsample=0.8, colsample_bytree=0.8, random_state=seed,
            scale_pos_weight=pos_weight, n_jobs=-1, eval_metric='logloss'
        )
        xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        oof_train[val_idx, 1] = xgb.predict_proba(X_val)[:, 1]
        oof_test[:, 1] += xgb.predict_proba(X_test_scaled)[:, 1] / n_splits
        
        # 3. CatBoost
        cat = CatBoostClassifier(
            iterations=2500, learning_rate=0.012, depth=8, l2_leaf_reg=5,
            random_state=seed, scale_pos_weight=pos_weight, verbose=False
        )
        cat.fit(X_train, y_train, eval_set=[(X_val, y_val)])
        cat_fold_pred = cat.predict_proba(X_test_scaled)[:, 1]
        oof_train[val_idx, 2] = cat.predict_proba(X_val)[:, 1]
        oof_test[:, 2] += cat_fold_pred / n_splits
        pure_cat_preds += cat_fold_pred / (n_splits * len(seeds))

    # Ridge Blender
    meta_model = RidgeClassifier(alpha=15.0, class_weight='balanced', random_state=seed)
    meta_model.fit(oof_train, y)
    
    decision_scores = meta_model.decision_function(oof_test)
    seed_preds = 1 / (1 + np.exp(-decision_scores))
    meta_preds_ensemble += seed_preds / len(seeds)

# ==========================================
# 5. GİZLİ SİLAH: LOGIT-SPACE MICRO-BLENDING
# ==========================================
# Olasılıkları kararlı kombinasyon için log-odds (logit) uzayına taşıyoruz
eps = 1e-5
final_preds_clipped = np.clip(meta_preds_ensemble, eps, 1 - eps)
pure_lgb_clipped = np.clip(pure_lgb_preds, eps, 1 - eps)
pure_cat_clipped = np.clip(pure_cat_preds, eps, 1 - eps)

logit_stacking = np.log(final_preds_clipped / (1 - final_preds_clipped))
logit_lgb = np.log(pure_lgb_clipped / (1 - pure_lgb_clipped))
logit_cat = np.log(pure_cat_clipped / (1 - pure_cat_clipped))

# Logit uzayında %90 - %5 - %5 geometrik harmanlama
final_logit = (logit_stacking * 0.90) + (logit_lgb * 0.05) + (logit_cat * 0.05)

# Tekrar pürüzsüz olasılık (Sigmoid) uzayına dönüş
final_preds = 1 / (1 + np.exp(-final_logit))

# ==========================================
# 6. ÇIKTI ÜRETME
# ==========================================
submission['class'] = final_preds
submission.to_csv('submission.csv', index=False)
print(" Kusursuz podyum dosyası 'submission.csv' hazır!")

 Podyum Operasyonu: Logit-Space Optimized Stacking Başlatıldı...
